# Flu Shot Learning - Notebook 15: repeated meta-blend

This experiment reuses the strongest model families with a fresh output name. It compares probability and rank blending using validation-only predictions and refits the selected target-specific ensemble for the DrivenData submission format.

The validation score is only a local estimate. Compare the generated CSV with the current best submission before uploading.

In [1]:
pip install iterative-stratification

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 2026
N_SPLITS = 5
TARGETS = ['h1n1_vaccine', 'seasonal_vaccine']
MODEL_NAMES = ['catboost', 'lightgbm', 'linear']

def find_data_dir():
    candidates = [Path.cwd() / 'data' / 'raw', Path.cwd() / 'data']
    candidates.extend(parent / 'data' / 'raw' for parent in Path.cwd().parents)
    candidates.extend([
        Path('/kaggle/input/datasets/barshamishra19/flu-shot-learning-dataset'),
        Path('/kaggle/input/flu-shot-learning-dataset'),
    ])
    for candidate in candidates:
        if (candidate / 'training_set_features.csv').exists():
            return candidate
    raise FileNotFoundError('Set DATA_DIR to the directory containing the four competition CSV files.')

DATA_DIR = find_data_dir()
train_features = pd.read_csv(DATA_DIR / 'training_set_features.csv')
train_labels = pd.read_csv(DATA_DIR / 'training_set_labels.csv')
test_features = pd.read_csv(DATA_DIR / 'test_set_features.csv')
y = train_labels[TARGETS].astype('int8')
assert train_features['respondent_id'].equals(train_labels['respondent_id'])
assert train_features['respondent_id'].is_unique and test_features['respondent_id'].is_unique
print(f'Data: {DATA_DIR.resolve()} | train={train_features.shape} | test={test_features.shape}')
display(y.mean().rename('positive_rate').to_frame())

Data: /kaggle/input/datasets/barshamishra19/flu-shot-learning-dataset | train=(26707, 36) | test=(26708, 36)


,positive_rate
h1n1_vaccine,0.212454
seasonal_vaccine,0.465608


In [3]:
def engineer_features(frame):
    result = frame.drop(columns=['respondent_id'], errors='ignore').copy()
    behavior = [c for c in result if c.startswith('behavioral_')]
    opinions = [c for c in result if c.startswith('opinion_')]
    risk = [c for c in opinions if '_risk' in c]
    result['missing_count'] = result.isna().sum(axis=1)
    result['missing_fraction'] = result.isna().mean(axis=1)
    result['behavior_score'] = result[behavior].sum(axis=1, min_count=1)
    result['behavior_available'] = result[behavior].notna().sum(axis=1)
    result['opinion_score'] = result[opinions].mean(axis=1)
    result['opinion_risk_score'] = result[risk].mean(axis=1)
    result['opinion_std'] = result[opinions].std(axis=1)
    result['doctor_recommendation_count'] = result[['doctor_recc_h1n1', 'doctor_recc_seasonal']].sum(axis=1, min_count=1)
    result['recommendation_gap'] = result['doctor_recc_seasonal'] - result['doctor_recc_h1n1']
    result['health_access_score'] = result[['health_insurance', 'health_worker']].sum(axis=1, min_count=1)
    result['health_worker_insured'] = result['health_worker'] * result['health_insurance']
    result['doctor_behavior_alignment'] = result['doctor_recommendation_count'] * result['behavior_score']
    result['doctor_opinion_alignment'] = result['doctor_recommendation_count'] * result['opinion_score']
    result['behavior_opinion_alignment'] = result['behavior_score'] * result['opinion_score']
    result['age_education'] = result['age_group'].astype('string') + '_' + result['education'].astype('string')
    result['doctor_pair'] = result['doctor_recc_h1n1'].astype('string') + '_' + result['doctor_recc_seasonal'].astype('string')
    ordinal_maps = {
        'age_group': {'18 - 34 Years': 1, '35 - 44 Years': 2, '45 - 54 Years': 3, '55 - 64 Years': 4, '65+ Years': 5},
        'education': {'< 12 Years': 1, '12 Years': 2, 'Some College': 3, 'College Graduate': 4},
        'income_poverty': {'Below Poverty': 1, '<= $75,000, Above Poverty': 2, '> $75,000': 3},
    }
    for column, mapping in ordinal_maps.items():
        result[column + '_ordinal'] = result[column].map(mapping)
    return result

X = engineer_features(train_features)
X_test = engineer_features(test_features)
categorical_columns = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
for column in categorical_columns:
    X[column] = X[column].fillna('__MISSING__').astype(str)
    X_test[column] = X_test[column].fillna('__MISSING__').astype(str)
cat_feature_indices = [X.columns.get_loc(column) for column in categorical_columns]
X_lgb, X_test_lgb = X.copy(), X_test.copy()
for column in categorical_columns:
    levels = pd.concat([X_lgb[column], X_test_lgb[column]], ignore_index=True).unique()
    dtype = pd.api.types.CategoricalDtype(categories=levels)
    X_lgb[column] = X_lgb[column].astype(dtype)
    X_test_lgb[column] = X_test_lgb[column].astype(dtype)
numeric_columns = X.select_dtypes(exclude=['object', 'category']).columns.tolist()
linear_preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)), ('scale', StandardScaler(with_mean=False))]), numeric_columns),
    ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('one_hot', OneHotEncoder(handle_unknown='ignore', sparse_output=True))]), categorical_columns),
], sparse_threshold=1.0)
print(f'Engineered features: {X.shape[1]} | categorical: {len(categorical_columns)}')

Engineered features: 54 | categorical: 14


In [4]:
def make_catboost(target, seed):
    depth, l2 = (7, 9) if target == 'h1n1_vaccine' else (6, 11)
    return CatBoostClassifier(depth=depth, iterations=2600, learning_rate=0.025, l2_leaf_reg=l2,
        loss_function='Logloss', eval_metric='AUC', random_seed=seed, verbose=False,
        allow_writing_files=False, thread_count=-1)

def make_lightgbm(target, seed):
    return LGBMClassifier(objective='binary', n_estimators=3500, learning_rate=0.015,
        num_leaves=24 if target == 'h1n1_vaccine' else 31, min_child_samples=45,
        subsample=0.9, colsample_bytree=0.85, reg_alpha=0.2, reg_lambda=2.0,
        random_state=seed, n_jobs=-1, verbosity=-1)

def make_linear():
    return Pipeline([('preprocessor', linear_preprocessor),
        ('classifier', LogisticRegression(C=0.18, solver='saga', penalty='l2', max_iter=3000,
                                          random_state=RANDOM_STATE, n_jobs=-1))])

folds = list(MultilabelStratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(X, y))
oof = {name: pd.DataFrame(0.0, index=X.index, columns=TARGETS) for name in MODEL_NAMES}
test_pred = {name: pd.DataFrame(0.0, index=X_test.index, columns=TARGETS) for name in MODEL_NAMES}
fold_oof = {name: {target: [] for target in TARGETS} for name in MODEL_NAMES}

for fold_number, (train_idx, valid_idx) in enumerate(folds, start=1):
    print(f'Fold {fold_number}/{N_SPLITS}')
    for target in TARGETS:
        seed = RANDOM_STATE + fold_number
        cat = make_catboost(target, seed)
        cat.fit(X.iloc[train_idx], y.iloc[train_idx][target], cat_features=cat_feature_indices,
                eval_set=(X.iloc[valid_idx], y.iloc[valid_idx][target]), use_best_model=True, early_stopping_rounds=180)
        cat_valid = cat.predict_proba(X.iloc[valid_idx])[:, 1]
        oof['catboost'].loc[valid_idx, target] = cat_valid
        test_pred['catboost'][target] += cat.predict_proba(X_test)[:, 1] / N_SPLITS

        lgb = make_lightgbm(target, seed + 100)
        lgb.fit(X_lgb.iloc[train_idx], y.iloc[train_idx][target], eval_set=[(X_lgb.iloc[valid_idx], y.iloc[valid_idx][target])],
                eval_metric='auc', categorical_feature=categorical_columns,
                callbacks=[early_stopping(180, verbose=False), log_evaluation(0)])
        lgb_valid = lgb.predict_proba(X_lgb.iloc[valid_idx])[:, 1]
        oof['lightgbm'].loc[valid_idx, target] = lgb_valid
        test_pred['lightgbm'][target] += lgb.predict_proba(X_test_lgb)[:, 1] / N_SPLITS

        linear = make_linear()
        linear.fit(X.iloc[train_idx], y.iloc[train_idx][target])
        linear_valid = linear.predict_proba(X.iloc[valid_idx])[:, 1]
        oof['linear'].loc[valid_idx, target] = linear_valid
        test_pred['linear'][target] += linear.predict_proba(X_test)[:, 1] / N_SPLITS

        fold_oof['catboost'][target].append((valid_idx, cat_valid))
        fold_oof['lightgbm'][target].append((valid_idx, lgb_valid))
        fold_oof['linear'][target].append((valid_idx, linear_valid))

model_scores = pd.DataFrame({name: [roc_auc_score(y[target], oof[name][target]) for target in TARGETS] for name in MODEL_NAMES}, index=TARGETS)
display(model_scores.assign(mean=model_scores.mean(axis=1)))

Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5


,catboost,lightgbm,linear,mean
h1n1_vaccine,0.871723,0.870521,0.862660,0.868301
seasonal_vaccine,0.864663,0.863583,0.855031,0.861092


In [5]:
def rank_normalize(values):
    return pd.Series(values).rank(method='average', pct=True).to_numpy()

def blend(values, weights, mode):
    if mode == 'rank':
        return sum(weights[name] * rank_normalize(values[name]) for name in MODEL_NAMES)
    return sum(weights[name] * np.asarray(values[name]) for name in MODEL_NAMES)

weight_grid = np.arange(0.0, 1.01, 0.1)
selected = {}
for target in TARGETS:
    candidates = []
    for cat_weight in weight_grid:
        for lgb_weight in weight_grid:
            linear_weight = round(1.0 - cat_weight - lgb_weight, 10)
            if linear_weight < 0:
                continue
            weights = {'catboost': float(cat_weight), 'lightgbm': float(lgb_weight), 'linear': linear_weight}
            for mode in ('probability', 'rank'):
                fold_scores = []
                for fold_number, (_, valid_idx) in enumerate(folds):
                    values = {name: oof[name].iloc[valid_idx][target].to_numpy() for name in MODEL_NAMES}
                    fold_scores.append(roc_auc_score(y.iloc[valid_idx][target], blend(values, weights, mode)))
                candidates.append((float(np.mean(fold_scores)), mode, weights, fold_scores))
    score, mode, weights, fold_scores = max(candidates, key=lambda item: item[0])
    selected[target] = {'score': score, 'mode': mode, 'weights': weights, 'fold_scores': fold_scores}
    print(target, mode, weights, 'mean fold AUC:', round(score, 6), 'folds:', np.round(fold_scores, 5))
print('Selected mean fold ROC AUC:', round(float(np.mean([item['score'] for item in selected.values()])), 6))

h1n1_vaccine probability {'catboost': 0.6000000000000001, 'lightgbm': 0.4, 'linear': np.float64(-0.0)} mean fold AUC: 0.872616 folds: [0.8688  0.86868 0.88283 0.87433 0.86845]
seasonal_vaccine rank {'catboost': 0.6000000000000001, 'lightgbm': 0.4, 'linear': np.float64(-0.0)} mean fold AUC: 0.865613 folds: [0.86384 0.86353 0.86906 0.86437 0.86727]
Selected mean fold ROC AUC: 0.869114


In [6]:
final_predictions = pd.DataFrame(index=X_test.index, columns=TARGETS, dtype=float)
for target in TARGETS:
    choice = selected[target]
    values = {name: test_pred[name][target].to_numpy() for name in MODEL_NAMES}
    final_predictions[target] = np.clip(blend(values, choice['weights'], choice['mode']), 0.0, 1.0)

submission = pd.DataFrame({'respondent_id': test_features['respondent_id'], **{target: final_predictions[target] for target in TARGETS}})
assert list(submission.columns) == ['respondent_id', *TARGETS]
assert submission['respondent_id'].equals(test_features['respondent_id'])
assert submission[TARGETS].notna().all().all()
output_dir = Path.cwd() / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
submission_path = output_dir / 'submission_notebook15_repeated_meta_blend.csv'
oof_path = output_dir / 'oof_notebook15_repeated_meta_blend.csv'
submission.to_csv(submission_path, index=False)
oof_export = pd.DataFrame({'respondent_id': train_features['respondent_id']})
for name in MODEL_NAMES:
    for target in TARGETS:
        oof_export[f'{name}_{target}'] = oof[name][target]
oof_export.to_csv(oof_path, index=False)
print(f'Saved submission: {submission_path.resolve()}')
print(f'Saved OOF predictions: {oof_path.resolve()}')
display(submission.head())

Saved submission: /kaggle/working/outputs/submission_notebook15_repeated_meta_blend.csv
Saved OOF predictions: /kaggle/working/outputs/oof_notebook15_repeated_meta_blend.csv


,respondent_id,h1n1_vaccine,seasonal_vaccine
0,26707,0.128410,0.301999
1,26708,0.028542,0.021290
2,26709,0.224789,0.669657
3,26710,0.660707,0.867568
4,26711,0.249989,0.532073
